In [1]:
# First we load the packages
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.formula.api as smf

# I. Data Preparation

In [2]:
# Read in master data
# Note this master dataset was created in the Week 2 assignment
Master = pd.read_csv("Master.csv")

In [3]:
print(Master.columns.tolist())

['Unnamed: 0', 'playerID', 'yearID', 'stint', 'G', 'AB', 'R', 'H', 'Doubles', 'Triples', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP', 'PA', 'OBP', 'SLG', 'SalYear', 'teamID', 'lgID', 'salary', 'lnSal', 'debutyr', 'Exp', 'Arb', 'Free', 'POS', 'Catch', 'Infld']


In [4]:
#1.3 Create the experience squared variable
Master['Exp2'] = Master.Exp ** 2

Batting Average = Hits/At-Bats

Isolated Power = Slugging Percentage – Batting Average

Eye = (Walks + Hit By Pitches)/Plate Appearances

In [5]:
#1.4 Calculate variables for batting average, isolated power, and eye 
#as defined above. Do not include IBB in walks.

Master['Avg'] = Master.H / Master.AB

Master['ISO'] = Master.SLG - Master.Avg

Master['Eye'] = (Master.BB + Master.HBP) / Master.PA

In [6]:
#1.5 Subset the data to only include seasons 1995-2015
Master = Master[(Master.SalYear >= 1995) & (Master.SalYear <= 2015)]

In [7]:
# Quiz 1 What is the highest single season "eye" measure for a player across all seasons in the data?

Master.Eye.max()

0.3905996758508914

In [8]:
Avg_ISO_team = (Master.groupby(['teamID', 'yearID'])['ISO']
                .mean()
                .reset_index())
Avg_ISO_team[Avg_ISO_team.ISO == Avg_ISO_team.ISO.max()]

,teamID,yearID,ISO
105,CHA,2008,0.209782


In [9]:
Seas_AVG = (Master.groupby(['yearID'])['Avg']
            .median()
            .reset_index())
Seas_AVG[Seas_AVG.Avg == Seas_AVG.Avg.max()]

,yearID,Avg
5,1999,0.277868


# II. Running Regressions for Each Season

In [10]:
# Write a function to run the Moneyball regression annually for free agents only
reg_formula = 'lnSal ~ Avg + ISO + Eye + PA + Exp + Exp2 + C(POS)' 
def MBExpandFA(Season):
    MB_Seas = Master[(Master.SalYear == Season) & (Master.Free == 1)]
    global lm
    lm = smf.ols(formula=reg_formula, data=MB_Seas).fit()
    return;

In [11]:
print(Master.columns.to_list())

['Unnamed: 0', 'playerID', 'yearID', 'stint', 'G', 'AB', 'R', 'H', 'Doubles', 'Triples', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP', 'PA', 'OBP', 'SLG', 'SalYear', 'teamID', 'lgID', 'salary', 'lnSal', 'debutyr', 'Exp', 'Arb', 'Free', 'POS', 'Catch', 'Infld', 'Exp2', 'Avg', 'ISO', 'Eye']


In [12]:
index = 0
lm_Results = [0]
for index in range(1, 21):
    lm_Results.append(index)
    index = index + 1
print(lm_Results)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [13]:
Season = 1995
i = 0
while Season <= 2014:
    MBExpandFA(Season)
    lm_Results[i] = lm
    i = i + 1
    Season = Season + 1

In [14]:
Season = 1995
lm_Season = ['1995']
for Season in range(1996, 2016):
    lm_Season.append(str(Season))
    Season = Season + 1

In [15]:
Pre_MB = lm_Season[:6]
MB_Period = lm_Season[6:13]
Post_MB = lm_Season[13:]

print(Pre_MB)
print(MB_Period)
print(Post_MB)

['1995', '1996', '1997', '1998', '1999', '2000']
['2001', '2002', '2003', '2004', '2005', '2006', '2007']
['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']


In [16]:
info_dict = {'R-squared' : lambda x: f"{x.rsquared:.2f}",
             'No. observations': lambda x: f"{int(x.nobs):d}"}

from statsmodels.iolib.summary2 import summary_col
PreMB_Out = summary_col(lm_Results[:6], 
                        model_names = Pre_MB, 
                        regressor_order=['AVG','Eye','ISO'],
                        stars=True,
                        info_dict = info_dict)
print(PreMB_Out)                       


                    1995      1996       1997       1998       1999       2000   
---------------------------------------------------------------------------------
Eye              2.4417    0.5096     2.0994     3.8206**   2.0140     1.0060    
                 (1.9611)  (2.0089)   (1.6643)   (1.6066)   (1.5924)   (1.5256)  
ISO              3.5872*** 4.9581***  2.7886***  3.1340***  2.4689***  3.5986*** 
                 (1.1991)  (1.3861)   (1.0009)   (1.1908)   (0.9210)   (0.9752)  
Intercept        9.9694*** 11.3986*** 12.2387*** 10.1032*** 10.3032*** 11.7979***
                 (1.1036)  (1.3914)   (1.0950)   (1.0611)   (0.9596)   (0.8532)  
C(POS)[T.2B]     -0.8179** -0.1286    -0.4298    -0.0151    -0.2160    -0.2726   
                 (0.3753)  (0.3466)   (0.2642)   (0.2872)   (0.2294)   (0.2175)  
C(POS)[T.3B]     -0.2760   -0.3602    -0.5173*   -0.0798    0.2404     -0.1103   
                 (0.3050)  (0.3140)   (0.2694)   (0.2479)   (0.2184)   (0.2205)  
C(POS)[T.C]    

In [17]:
MB_Out = summary_col(lm_Results[6:13], 
                        model_names = MB_Period, 
                        regressor_order=['AVG','Eye','ISO'],
                        stars=True,
                        info_dict = info_dict)
print(MB_Out)    


                    2001       2002       2003       2004       2005       2006       2007   
---------------------------------------------------------------------------------------------
Eye              -3.0326*   1.7225     3.0337     9.3959***  2.8751     2.9151     4.2304**  
                 (1.6224)   (2.0778)   (2.1865)   (2.1082)   (1.8720)   (2.0959)   (1.8341)  
ISO              5.0715***  2.7926**   1.4127     1.8573     3.1916***  2.6517**   3.0405*** 
                 (1.0084)   (1.3380)   (1.3554)   (1.2495)   (1.2057)   (1.2112)   (1.0456)  
Intercept        11.2572*** 10.3452*** 10.0969*** 10.7180*** 10.4152*** 10.6209*** 10.8980***
                 (0.7926)   (1.0372)   (1.1665)   (1.1329)   (0.9773)   (0.9818)   (0.7737)  
C(POS)[T.2B]     0.1785     -0.0029    -0.2719    0.0510     -0.4057    -0.1014    -0.2304   
                 (0.2204)   (0.2865)   (0.2921)   (0.2977)   (0.2602)   (0.2517)   (0.2269)  
C(POS)[T.3B]     0.2367     0.2321     0.0073     -0.0236  

In [18]:
Post_MB = summary_col(lm_Results[13:20], 
                        model_names = Post_MB, 
                        regressor_order=['AVG','Eye','ISO'],
                        stars=True,
                        info_dict = info_dict)
print(Post_MB)    


                    2008       2009       2010      2011      2012       2013       2014   
-------------------------------------------------------------------------------------------
Eye              3.4829     4.3845**   6.2334*** 4.0906    2.6060     4.1774*    5.8097**  
                 (2.1335)   (2.1487)   (2.3060)  (2.7579)  (2.5512)   (2.3415)   (2.5174)  
ISO              3.1244**   1.6764     2.5375    3.1109**  3.2232**   2.7647*    2.9424**  
                 (1.2600)   (1.4667)   (1.5673)  (1.4932)  (1.6000)   (1.4094)   (1.3941)  
Intercept        11.6518*** 10.7617*** 8.9933*** 9.8735*** 12.8252*** 12.8365*** 10.5028***
                 (0.9852)   (0.9933)   (1.2090)  (1.2633)  (1.0809)   (1.2736)   (1.1818)  
C(POS)[T.2B]     0.0278     -0.1089    0.0035    0.2046    0.1738     -0.5499**  0.3901    
                 (0.2562)   (0.3495)   (0.3420)  (0.3225)  (0.3226)   (0.2748)   (0.2568)  
C(POS)[T.3B]     0.5582**   0.3158     0.4651    -0.0762   0.4312     -0.2036  

# III. Running the Pooled Regression

In [31]:
Master_Free = Master[Master.Free == 1].copy()
Master_Free['PostMB'] = np.where(Master_Free['SalYear'] >= 2004, 1, 0)
Master_Free.shape

(2841, 41)

In [32]:
reg_formula1 = 'lnSal ~ Avg + ISO + Eye + PA + Exp + Exp2 + C(POS) +' 
reg_formula2 = 'PostMB*(Avg + ISO + Eye + PA + Exp + Exp2 + C(POS))'
Pooled_lm = smf.ols(formula = reg_formula1 +
                              reg_formula2, 
                    data=Master_Free).fit()

Pooled_lm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  lnSal   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.516
Method:                 Least Squares   F-statistic:                     122.2
Date:                Tue, 11 Jan 2022   Prob (F-statistic):               0.00
Time:                        21:00:44   Log-Likelihood:                -3313.7
No. Observations:                2841   AIC:                             6679.
Df Residuals:                    2815   BIC:                             6834.
Df Model:                          25                                         
Covariance Type:            nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              10.9239      0.356     30.691      0.000      10.226      11.622
C(POS)[T.2B]           -0.2026      0.097     -2.091      0.037      -0.392      -0.013
C(POS)[T.3B]           -0.0877      0.092     -0.955      0.340      -0.268       0.092
C(POS)[T.C]             0.0668      0.092      0.725      0.469      -0.114       0.248
C(POS)[T.DH]           -0.1500      0.113     -1.333      0.183      -0.371       0.071
C(POS)[T.OF]           -0.0288      0.073     -0.394      0.694      -0.172       0.115
C(POS)[T.SS]            0.1110      0.100      1.106      0.269      -0.086       0.308
Avg                     1.9829      0.774      2.561      0.010       0.465       3.501
ISO                     3.5023      0.390      8.989      0.000       2.738       4.266
Eye                     1.3575      0.623      2.178      0.030       0.135       2.580
PA                      0.0036      0.000     22.819      0.000       0.003       0.004
Exp                     0.1683      0.051      3.323      0.001       0.069       0.268
Exp2                   -0.0077      0.002     -3.522      0.000      -0.012      -0.003
PostMB                  0.6327      0.463      1.367      0.172      -0.275       1.540
PostMB:C(POS)[T.2B]     0.0952      0.128      0.746      0.456      -0.155       0.346
PostMB:C(POS)[T.3B]     0.2705      0.121      2.232      0.026       0.033       0.508
PostMB:C(POS)[T.C]      0.0250      0.121      0.206      0.837      -0.213       0.263
PostMB:C(POS)[T.DH]     0.1985      0.149      1.332      0.183      -0.094       0.491
PostMB:C(POS)[T.OF]     0.1558      0.097      1.602      0.109      -0.035       0.346
PostMB:C(POS)[T.SS]     0.0129      0.132      0.098      0.922      -0.247       0.273
PostMB:Avg              0.9981      1.035      0.964      0.335      -1.032       3.028
PostMB:ISO             -1.3514      0.555     -2.436      0.015      -2.439      -0.264
PostMB:Eye              2.4039      0.894      2.690      0.007       0.652       4.156
PostMB:PA           -2.091e-05      0.000     -0.101      0.919      -0.000       0.000
PostMB:Exp             -0.0722      0.065     -1.109      0.268      -0.200       0.055
PostMB:Exp2             0.0032      0.003      1.169      0.242      -0.002       0.009
==============================================================================
Omnibus:                       15.145   Durbin-Watson:                   1.331
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               20.745
Skew:                           0.027   Prob(JB):                     3.13e-05
Kurtosis:                       3.415   Cond. No.                     5.21e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance ma